# Introduction à PySpark – Lazy Evaluation & Parquet

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tanouar/1to1code/blob/main/pySpark/notebooks/03_traitement_parquet.ipynb)

Notebook conçu pour **Google Colab**.  
On explore la lecture d'un fichier Parquet réel (Base Adresse Nationale — ~500 MB),  
le chaînage de transformations **lazy**, et leur déclenchement via des **actions**.

> **SparkUI** : accessible après la cellule de setup via le lien affiché.


In [3]:
# Installation + setup (Colab)
!pip install -q pyspark findspark
!git clone --filter=blob:none --sparse https://github.com/tanouar/1to1code.git -q \
  && cd 1to1code && git sparse-checkout set pySpark -q

import os, sys
PYSPARK_TOOLS = os.path.join('/content', '1to1code', 'pySpark')
if PYSPARK_TOOLS not in sys.path:
    sys.path.insert(0, PYSPARK_TOOLS)

from sparktools import setup_colab, download_ban
from pyspark.sql import functions as F
import time

spark, monitor, helper = setup_colab("PySpark - Traitement Parquet")


fatal: destination path '1to1code' already exists and is not an empty directory.
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

✓ SparkUI ouverte → cliquez sur le lien ci-dessus
✓ Spark 4.0.2  |  App : PySpark - Traitement Parquet
✓ Memory : 4g  |  Shuffle partitions : 8
✓ SparkJobMonitor et SparkHelper initialisés


## 1. Téléchargement de la Base Adresse Nationale (BAN)

`download_ban()` télécharge le fichier Parquet (~500 MB) depuis data.gouv.fr.


In [4]:
parquet_path = download_ban()   # télécharge dans /content/adresse_france.parquet
print(f"Fichier prêt : {parquet_path}")


⏳ Téléchargement BAN (~500 MB) — quelques minutes...
✓ BAN téléchargée → /content/adresse_france.parquet (478 MB)
Fichier prêt : /content/adresse_france.parquet


## 2. Lecture Lazy du Parquet

`spark.read.parquet()` crée un **plan d'exécution** mais ne lit aucune donnée.  
→ SparkUI : aucun job ne doit apparaître après cette cellule.


In [5]:
start = time.time()
df_ban = spark.read.parquet(parquet_path)
elapsed = time.time() - start
print(f"Plan créé en {elapsed:.4f}s — aucune donnée lue.")
print("→ SparkUI : vérifiez qu'aucun job n'apparaît dans l'onglet Jobs.")


Plan créé en 3.3942s — aucune donnée lue.
→ SparkUI : vérifiez qu'aucun job n'apparaît dans l'onglet Jobs.


## 3. Exploration du Schéma (lazy)

Le schéma est disponible via les métadonnées Parquet — **sans lire une seule ligne**.


In [6]:
df_ban.printSchema()
print(f"\n{len(df_ban.columns)} colonnes disponibles — schéma lu depuis les métadonnées Parquet, sans toucher aux données.")


root
 |-- id: string (nullable = true)
 |-- id_fantoir: string (nullable = true)
 |-- numero: long (nullable = true)
 |-- rep: string (nullable = true)
 |-- nom_voie: string (nullable = true)
 |-- code_postal: string (nullable = true)
 |-- code_insee: string (nullable = true)
 |-- nom_commune: string (nullable = true)
 |-- code_insee_ancienne_commune: string (nullable = true)
 |-- nom_ancienne_commune: string (nullable = true)
 |-- type_position: string (nullable = true)
 |-- alias: string (nullable = true)
 |-- nom_ld: string (nullable = true)
 |-- libelle_acheminement: string (nullable = true)
 |-- nom_afnor: string (nullable = true)
 |-- source_position: string (nullable = true)
 |-- source_nom_voie: string (nullable = true)
 |-- certification_commune: long (nullable = true)
 |-- cad_parcelles: string (nullable = true)
 |-- geom: binary (nullable = true)
 |-- h3_7: decimal(20,0) (nullable = true)
 |-- quadkey: string (nullable = true)


22 colonnes disponibles — schéma lu depuis les

## 4. Chaînage de Transformations (lazy)

On enchaîne 5 transformations sur l'ensemble du dataset.  
→ SparkUI : toujours aucun job. Spark construit un **DAG** sans calculer.


In [7]:
start = time.time()

df_sorted = (
    df_ban
    .select("id", "numero", "nom_voie", "nom_commune", "code_postal")
    .filter(F.col("code_postal").startswith("75"))
    .withColumn("adresse_complete",
                F.concat_ws(" ", "numero", "nom_voie", "code_postal", "nom_commune"))
    .dropDuplicates(["adresse_complete"])
    .orderBy("code_postal", "nom_voie")
)

elapsed = time.time() - start
print(f"5 transformations enchaînées en {elapsed:.4f}s — toujours lazy.")
print("→ SparkUI : aucun job. Spark a construit un DAG mais n'a rien calculé.")


5 transformations enchaînées en 0.3182s — toujours lazy.
→ SparkUI : aucun job. Spark a construit un DAG mais n'a rien calculé.


## 5. Première Action – show()

`show()` est une **action** : elle force l'exécution du plan entier.  
→ SparkUI : un job apparaît enfin dans l'onglet **Jobs**.


In [8]:
monitor.execute_and_monitor(
    lambda: df_sorted.show(10, truncate=False),
    "JOB 1: show() – 10 premières adresses de Paris"
)
print("→ SparkUI : examinez le plan SQL, cherchez 'PushedFilters' (predicate pushdown).")



🔵 JOB 1: show() – 10 premières adresses de Paris
📌 Tracking ID: f8ae6b3b
+---------------------+------+-----------------------+------------------------+-----------+---------------------------------------------------------+
|id                   |numero|nom_voie               |nom_commune             |code_postal|adresse_complete                                         |
+---------------------+------+-----------------------+------------------------+-----------+---------------------------------------------------------+
|75101_ofangt_00012_s1|12    |12S1 Place du Carrousel|Paris 1er Arrondissement|75001      |12 12S1 Place du Carrousel 75001 Paris 1er Arrondissement|
|75101_adsmy0_00001_s1|1     |1S1 Place du Carrousel |Paris 1er Arrondissement|75001      |1 1S1 Place du Carrousel 75001 Paris 1er Arrondissement  |
|75101_fztbae_00002_s1|2     |2S1 Place du Carrousel |Paris 1er Arrondissement|75001      |2 2S1 Place du Carrousel 75001 Paris 1er Arrondissement  |
|75101_5paf6z_00004_w1|4  

## 6. Statistiques Globales

On déclenche plusieurs actions sur l'ensemble du dataset.  
→ SparkUI : observez le cumul de jobs dans l'onglet **Jobs**.


In [9]:
result = monitor.execute_and_monitor(
    lambda: df_ban.count(),
    "JOB 2: Count total BAN"
)
print(f"\n{result:,} adresses en France")

monitor.execute_and_monitor(
    lambda: (
        df_ban
        .withColumn("departement", F.substring("code_postal", 1, 2))
        .groupBy("departement")
        .agg(F.count("*").alias("nb_adresses"))
        .orderBy(F.col("nb_adresses").desc())
        .limit(10)
        .show(truncate=False)
    ),
    "JOB 3: Top 10 départements"
)



🔵 JOB 2: Count total BAN
📌 Tracking ID: e5354aa9

✅ SUCCESS
⏱️  Durée: 1.56s
📊 Spark Job ID(s): [4, 3]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/

26,045,333 adresses en France

🔵 JOB 3: Top 10 départements
📌 Tracking ID: 01f3783a
+-----------+-----------+
|departement|nb_adresses|
+-----------+-----------+
|59         |1014683    |
|33         |748993     |
|97         |740928     |
|62         |697765     |
|44         |572933     |
|29         |494094     |
|13         |471566     |
|76         |470813     |
|31         |460327     |
|34         |456267     |
+-----------+-----------+


✅ SUCCESS
⏱️  Durée: 11.64s
📊 Spark Job ID(s): [6, 5]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/


## 7. Récapitulatif : Transformations vs Actions

| Transformations (lazy — pas de job) | Actions (déclenchent un job) |
|---|---|
| `select()` | `show()` |
| `filter()` / `where()` | `count()` |
| `withColumn()` | `collect()` / `take(n)` |
| `groupBy()` | `describe()` |
| `orderBy()` / `sort()` | `write()` |
| `join()` | `toPandas()` |
| `distinct()` / `dropDuplicates()` | `first()` / `head()` |

→ SparkUI : un nouveau **job** apparaît à chaque action. Observez le nombre de jobs qui s'accumulent dans l'onglet Jobs.


## 8. Optimisations Parquet

Spark applique automatiquement deux optimisations sur les fichiers Parquet :
- **Predicate pushdown** : filtres appliqués au niveau du fichier (avant lecture)
- **Column pruning** : Spark lit uniquement les colonnes nécessaires


In [11]:
# Predicate pushdown : filtre appliqué au niveau Parquet
result = monitor.execute_and_monitor(
    lambda: df_ban.filter(F.col("code_postal") == "75001").count(),
    "JOB 4: Predicate pushdown – adresses 75001"
)
print(f"{result:,} adresses dans le 75001 — Spark a filtré dans le fichier Parquet directement.")

# Column pruning : seulement 2 colonnes lues
result = monitor.execute_and_monitor(
    lambda: (
        df_ban.select("nom_commune", "code_postal")
        .filter(F.col("code_postal").startswith("13"))
        .distinct()
        .count()
    ),
    "JOB 5: Column pruning – communes Bouches-du-Rhône"
)
print(f"{result} communes dans le 13 — seulement 2 colonnes lues sur {len(df_ban.columns)}.")

# Explain (physical plan)
print("\n--- Plan d'exécution ---")
df_ban.filter(F.col("code_postal").startswith("69")) \
    .select("nom_commune", "nom_voie") \
    .groupBy("nom_commune").agg(F.count("*").alias("n")) \
    .explain("simple")



🔵 JOB 4: Predicate pushdown – adresses 75001
📌 Tracking ID: 71b0e3c7

✅ SUCCESS
⏱️  Durée: 2.27s
📊 Spark Job ID(s): [8, 7]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/
3,309 adresses dans le 75001 — Spark a filtré dans le fichier Parquet directement.

🔵 JOB 5: Column pruning – communes Bouches-du-Rhône
📌 Tracking ID: 86ad55cd

✅ SUCCESS
⏱️  Durée: 3.34s
📊 Spark Job ID(s): [11, 10, 9]
📦 Stages: 6 | Tasks: 15
🌐 Spark UI: http://localhost:4040/jobs/
154 communes dans le 13 — seulement 2 colonnes lues sur 22.

--- Plan d'exécution ---
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[nom_commune#7], functions=[count(1)])
   +- Exchange hashpartitioning(nom_commune#7, 8), ENSURE_REQUIREMENTS, [plan_id=347]
      +- HashAggregate(keys=[nom_commune#7], functions=[partial_count(1)])
         +- Project [nom_commune#7]
            +- Filter (isnotnull(code_postal#5) AND StartsWith(code_postal#5, 69))
               +- FileScan parquet [code_postal

## 9. Exercices Pratiques


In [12]:
# Exercice 1 — Top 10 noms de rues les plus fréquents
monitor.execute_and_monitor(
    lambda: df_ban.groupBy("nom_voie")
        .agg(F.count("*").alias("nb"))
        .orderBy(F.col("nb").desc())
        .limit(10)
        .show(truncate=False),
    "EXERCICE 1: Top 10 noms de rues"
)

# Exercice 2 — Communes avec le plus de codes postaux différents
monitor.execute_and_monitor(
    lambda: df_ban.groupBy("nom_commune")
        .agg(F.countDistinct("code_postal").alias("nb_codes"))
        .filter(F.col("nb_codes") > 1)
        .orderBy(F.col("nb_codes").desc())
        .limit(10)
        .show(truncate=False),
    "EXERCICE 2: Communes multi-codes postaux"
)

# Exercice 3 — Stats par département
monitor.execute_and_monitor(
    lambda: df_ban.withColumn("dept", F.substring("code_postal", 1, 2))
        .groupBy("dept")
        .agg(
            F.count("*").alias("nb_adresses"),
            F.countDistinct("nom_commune").alias("nb_communes"),
            F.countDistinct("nom_voie").alias("nb_voies_uniques")
        )
        .orderBy(F.col("nb_adresses").desc())
        .limit(15)
        .show(truncate=False),
    "EXERCICE 3: Stats par département"
)



🔵 EXERCICE 1: Top 10 noms de rues
📌 Tracking ID: 2a2a6afd
+--------------------+------+
|nom_voie            |nb    |
+--------------------+------+
|Grande Rue          |154834|
|Rue Principale      |81117 |
|Rue de l'Eglise     |60429 |
|Rue de la Gare      |54014 |
|Rue Pasteur         |53482 |
|Rue du Moulin       |53281 |
|Rue de la République|48979 |
|Rue Victor Hugo     |48773 |
|Rue de la Mairie    |45573 |
|Rue Jean Jaurès     |39473 |
+--------------------+------+


✅ SUCCESS
⏱️  Durée: 41.43s
📊 Spark Job ID(s): [13, 12]
📦 Stages: 3 | Tasks: 10
🌐 Spark UI: http://localhost:4040/jobs/

🔵 EXERCICE 2: Communes multi-codes postaux
📌 Tracking ID: bdfd08ba
+--------------+--------+
|nom_commune   |nb_codes|
+--------------+--------+
|Saint-Paul    |13      |
|Sainte-Colombe|12      |
|Saint-Sauveur |11      |
|Beaulieu      |10      |
|Saint-Aubin   |10      |
|Saint-Pierre  |10      |
|Saint-Marcel  |9       |
|Saint-Michel  |9       |
|Sainte-Marie  |9       |
|Haguenau      |9  

## 10. Cache et Performance

On compare les temps d'exécution avec et sans cache.  
→ SparkUI → **Storage** : observez les données en mémoire après `cache()`.


In [13]:
df_idf = df_ban.filter(
    F.col("code_postal").startswith("75") |
    F.col("code_postal").startswith("77") |
    F.col("code_postal").startswith("78") |
    F.col("code_postal").startswith("91") |
    F.col("code_postal").startswith("92") |
    F.col("code_postal").startswith("93") |
    F.col("code_postal").startswith("94") |
    F.col("code_postal").startswith("95")
)

# Sans cache
result1 = monitor.execute_and_monitor(lambda: df_idf.count(), "JOB: Count Île-de-France (sans cache)")
print(f"{result1:,} adresses en Île-de-France")

# Mise en cache
df_idf.cache()

# Avec cache (premier passage — populate)
monitor.execute_and_monitor(lambda: df_idf.count(), "JOB: Count IDF (mise en cache)")

# Avec cache (depuis le cache)
monitor.execute_and_monitor(lambda: df_idf.count(), "JOB: Count IDF (depuis cache)")

print("→ SparkUI → Storage : observez les données en cache et la mémoire utilisée.")
print("→ Comparez les durées des 3 jobs dans l'onglet Jobs.")

df_idf.unpersist()
print("Cache libéré.")



🔵 JOB: Count Île-de-France (sans cache)
📌 Tracking ID: e05b277e

✅ SUCCESS
⏱️  Durée: 5.69s
📊 Spark Job ID(s): [21, 20]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/
2,186,664 adresses en Île-de-France

🔵 JOB: Count IDF (mise en cache)
📌 Tracking ID: f406ea43

✅ SUCCESS
⏱️  Durée: 87.35s
📊 Spark Job ID(s): [24, 23, 22]
📦 Stages: 4 | Tasks: 13
🌐 Spark UI: http://localhost:4040/jobs/

🔵 JOB: Count IDF (depuis cache)
📌 Tracking ID: 8438904a

✅ SUCCESS
⏱️  Durée: 0.69s
📊 Spark Job ID(s): [26, 25]
📦 Stages: 3 | Tasks: 9
🌐 Spark UI: http://localhost:4040/jobs/
→ SparkUI → Storage : observez les données en cache et la mémoire utilisée.
→ Comparez les durées des 3 jobs dans l'onglet Jobs.
Cache libéré.


In [14]:
# ============================================
# PARTIE 11 : Repartitionnement et Écriture
# ============================================

print("\n💾 PARTIE 11 : Repartitionnement et Écriture Parquet")
print("="*60)

# Filtrer sur une région
df_bretagne = df_ban.filter(
    F.col("code_postal").startswith("22") |
    F.col("code_postal").startswith("29") |
    F.col("code_postal").startswith("35") |
    F.col("code_postal").startswith("56")
).withColumn("departement", F.substring("code_postal", 1, 2))

output_path = "/tmp/spark_demo/ban_bretagne_parquet"

print("\n1️⃣ Écriture partitionnée par département")

result = monitor.execute_and_monitor(
    lambda: df_bretagne.write
        .mode("overwrite")
        .partitionBy("departement")
        .parquet(output_path),
    "JOB 10: Écriture Parquet partitionnée (Bretagne)"
)

print(f"\n✓ Données écrites dans: {output_path}")

print("\n📁 Structure des partitions:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = ' ' * 2 * level
    folder = os.path.basename(root)
    if folder:
        print(f'{indent}{folder}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for file in files[:3]:
            size = os.path.getsize(os.path.join(root, file)) / (1024**2)
            print(f'{subindent}{file} ({size:.2f} MB)')
        if len(files) > 3:
            print(f'{subindent}... et {len(files) - 3} autres fichiers')

print("\n2️⃣ Lecture avec partition pruning")

result = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path)
        .filter(F.col("departement") == "29")
        .count(),
    "JOB 11: Lecture avec partition pruning (Finistère)"
)

print(f"\n✓ {result:,} adresses dans le Finistère (29)")
print("💡 Spark a lu SEULEMENT le dossier departement=29/")


💾 PARTIE 11 : Repartitionnement et Écriture Parquet

1️⃣ Écriture partitionnée par département

🔵 JOB 10: Écriture Parquet partitionnée (Bretagne)
📌 Tracking ID: 9edf7602

✅ SUCCESS
⏱️  Durée: 64.67s
📊 Spark Job ID(s): [27]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/

✓ Données écrites dans: /tmp/spark_demo/ban_bretagne_parquet

📁 Structure des partitions:
ban_bretagne_parquet/
  ._SUCCESS.crc (0.00 MB)
  _SUCCESS (0.00 MB)
  departement=56/
    .part-00002-bd233eee-74c0-44bb-9ba4-e4d6b53e9f7b.c000.snappy.parquet.crc (0.09 MB)
    part-00002-bd233eee-74c0-44bb-9ba4-e4d6b53e9f7b.c000.snappy.parquet (11.85 MB)
  departement=35/
    part-00001-bd233eee-74c0-44bb-9ba4-e4d6b53e9f7b.c000.snappy.parquet (12.23 MB)
    .part-00001-bd233eee-74c0-44bb-9ba4-e4d6b53e9f7b.c000.snappy.parquet.crc (0.10 MB)
  departement=22/
    .part-00000-bd233eee-74c0-44bb-9ba4-e4d6b53e9f7b.c000.snappy.parquet.crc (0.08 MB)
    part-00000-bd233eee-74c0-44bb-9ba4-e4d6b53e9f7b.c000.snappy.parquet

In [ ]:
# ============================================
# PARTIE 11 BIS : Partitionnement (version concise)
# ============================================

import os
import glob
import time

print("\nPARTIE 11 BIS - Partitionnement Parquet (synthese)")

# 1) Jeu d'essai restreint pour comparaison
# On garde 3 departements pour observer clairement le partition pruning.
df_sample = (
    df_ban
    .filter(
        F.col("code_postal").startswith("75") |
        F.col("code_postal").startswith("92") |
        F.col("code_postal").startswith("93")
    )
    .withColumn("departement", F.substring("code_postal", 1, 2))
)

path_non_partitioned = "/tmp/spark_demo/ban_non_partitioned"
path_partitioned = "/tmp/spark_demo/ban_partitioned"

# 2) Ecriture non partitionnee vs partitionnee
monitor.execute_and_monitor(
    lambda: df_sample.write.mode("overwrite").parquet(path_non_partitioned),
    "ECRITURE: non partitionnee"
)

monitor.execute_and_monitor(
    lambda: (
        df_sample.write.mode("overwrite")
        .partitionBy("departement")
        .parquet(path_partitioned)
    ),
    "ECRITURE: partitionnee par departement"
)

# 3) Taille de donnees ecrites (fichiers parquet uniquement)
def parquet_size_mb(path):
    total = sum(
        os.path.getsize(os.path.join(root, file))
        for root, _, files in os.walk(path)
        for file in files
        if file.endswith(".parquet")
    )
    return total / (1024 ** 2)

size_non = parquet_size_mb(path_non_partitioned)
size_part = parquet_size_mb(path_partitioned)

print(f"- Taille non partitionnee : {size_non:.2f} MB")
print(f"- Taille partitionnee     : {size_part:.2f} MB")

# 4) Lecture filtree: impact du partition pruning
start = time.time()
count_non = (
    spark.read.parquet(path_non_partitioned)
    .filter(F.col("departement") == "75")
    .count()
)
t_non = time.time() - start

start = time.time()
count_part = (
    spark.read.parquet(path_partitioned)
    .filter(F.col("departement") == "75")
    .count()
)
t_part = time.time() - start

print(f"- Adresses dept 75 (non partitionne) : {count_non:,} | {t_non:.2f}s")
print(f"- Adresses dept 75 (partitionne)     : {count_part:,} | {t_part:.2f}s")

if t_part > 0:
    print(f"- Gain approx. (x) : {t_non / t_part:.2f}")

# 5) Nombre de fichiers par partition (sanity check)
for dept in ["75", "92", "93"]:
    dept_path = os.path.join(path_partitioned, f"departement={dept}")
    n_files = len(glob.glob(os.path.join(dept_path, "*.parquet"))) if os.path.exists(dept_path) else 0
    print(f"- departement={dept} -> {n_files} fichier(s) parquet")

print("\nA retenir: partitionner sur une colonne frequemment filtree accelere les lectures selectives.")


🔍 PARTIE 11 BIS : Le Partitionnement Parquet Expliqué en Détail

📚 QU'EST-CE QUE LE PARTITIONNEMENT ?
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Le partitionnement divise les données en sous-dossiers selon les valeurs
d'une ou plusieurs colonnes.

Structure classique:
    data/
    ├── departement=75/
    │   ├── part-00000-xxx.snappy.parquet
    │   └── part-00001-xxx.snappy.parquet
    ├── departement=92/
    │   └── part-00000-xxx.snappy.parquet
    └── _SUCCESS

Avantages:
  ✓ PARTITION PRUNING : Spark lit seulement les dossiers nécessaires
  ✓ PERFORMANCE : Évite de scanner toutes les données
  ✓ ORGANISATION : Structure logique et claire
  ✓ COMPATIBILITÉ : Standard Hive/Hadoop


1️⃣ EXPÉRIMENTATION : Partitionné vs Non-Partitionné

📝 Écriture NON partitionnée...

🔵 ÉCRITURE : Sans partitionnement
📌 Tracking ID: 67bdb41a

✅ SUCCESS
⏱️  Durée: 19.32s
📊 Spark Job ID(s): [31]
📦 Stages: 1 | Tasks: 4
🌐 Spark UI: http://localhost:4040/jobs/

📝 Écriture PARTITIONNÉE par département...

🔵

In [ ]:
# ============================================
# RÉSUMÉ ET HISTORIQUE (version concise)
# ============================================

print("\nRESUME RAPIDE")
print("- Transformations: lazy (pas d'execution immediate)")
print("- Actions: declenchent les jobs")
print("- Parquet: column pruning + predicate pushdown")
print("- Partitionnement: utile pour les lectures filtrees")

# Historique et stats (utile pour auto-evaluation)
monitor.show_history()
stats = monitor.get_job_stats()

print("\nSTATISTIQUES")
print(f"- Total jobs : {stats['total_jobs']}")
print(f"- Reussis    : {stats['success']}")
print(f"- Echoues    : {stats['failed']}")
print(f"- Duree tot. : {stats['total_duration']:.2f}s")
print(f"- Duree moy. : {stats['avg_duration']:.2f}s")

print("\nNotebook termine.")


📚 RÉSUMÉ DES CONCEPTS CLÉS

🎯 LAZY EVALUATION
   ✓ Les transformations ne s'exécutent PAS immédiatement
   ✓ Spark construit un DAG (graphe d'exécution)
   ✓ Les actions déclenchent l'exécution complète
   ✓ Permet d'optimiser avant de calculer

🔄 TRANSFORMATIONS (lazy)
   • select, filter, withColumn, groupBy, join
   • Retournent un nouveau DataFrame
   • AUCUNE donnée lue ou calculée
   • S'enchaînent instantanément

⚡ ACTIONS (eager - déclenchent l'exécution)
   • show, count, collect, write
   • Exécutent TOUT le plan d'exécution
   • Lisent et calculent les données
   • Retournent un résultat

⚙️ OPTIMISATIONS SPARK
   ✓ Predicate Pushdown : filtres au niveau Parquet
   ✓ Column Pruning : lecture sélective de colonnes
   ✓ Partition Pruning : lecture sélective de dossiers
   ✓ Catalyst Optimizer : réorganise le plan

📦 PARQUET
   ✓ Format colonnaire (parfait pour l'analytique)
   ✓ Compression efficace (gain 5-10x vs CSV)
   ✓ Métadonnées riches (schéma, stats)
   ✓ Partitionnem